<a href="https://colab.research.google.com/github/blbl-blbl/study/blob/main/PyTorch/01_oxford_pets/06_augmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Oxford-IIIT Pet — Data Augmentation

Текущая тема курса: мягкие train-аугментации (`RandomResizedCrop`, `RandomHorizontalFlip`) при неизменной validation-preprocessing и отдельные DataLoader.

> Этот notebook самодостаточен: его можно запускать сверху вниз в чистом Google Colab. Он не требует выполнения других notebook-файлов проекта.

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import random
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import OxfordIIITPet
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

raw_dataset = OxfordIIITPet(
    root="data",
    split="trainval",
    target_types="category",
    download=True,
)

labels = np.array([raw_dataset[i][1] for i in range(len(raw_dataset))])

train_indices, val_indices = train_test_split(
    np.arange(len(raw_dataset)),
    test_size=0.2,
    random_state=42,
    stratify=labels,
)

class_names = raw_dataset.classes
weights = ResNet18_Weights.IMAGENET1K_V1
resnet_transform = weights.transforms()

resnet_train_source = OxfordIIITPet(
    root="data",
    split="trainval",
    target_types="category",
    transform=resnet_transform,
    download=False,
)

resnet_val_source = OxfordIIITPet(
    root="data",
    split="trainval",
    target_types="category",
    transform=resnet_transform,
    download=False,
)

resnet_train_dataset = Subset(resnet_train_source, train_indices.tolist())
resnet_val_dataset = Subset(resnet_val_source, val_indices.tolist())

resnet_train_loader = DataLoader(
    resnet_train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(42),
)

resnet_val_loader = DataLoader(
    resnet_val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

loss_fn = nn.CrossEntropyLoss()

print("Train:", len(resnet_train_dataset))
print("Validation:", len(resnet_val_dataset))
print("Classes:", len(class_names))


## Мягкая аугментация только для train

Validation остаётся детерминированной. Это позволяет проверять эффект аугментации, не меняя способ оценки модели.

In [ ]:
resnet_train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        224,
        scale=(0.8, 1.0)
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# Validation-преобразования оставим прежними и детерменированными
resnet_val_transform = weights.transforms()

In [ ]:
aug_train_source = OxfordIIITPet(
    root="data",
    split="trainval",
    target_types="category",
    transform=resnet_train_transform,
    download=False,
)

aug_val_source = OxfordIIITPet(
    root="data",
    split="trainval",
    target_types="category",
    transform=resnet_val_transform,
    download=False,
)


aug_train_dataset = Subset(
    aug_train_source,
    train_indices.tolist()
)

aug_val_dataset = Subset(
    aug_val_source,
    val_indices.tolist()
)


aug_train_loader = DataLoader(
    aug_train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(42)
)

aug_val_loader = DataLoader(
    aug_val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

### Визуальная проверка

Проверяем несколько изображений после случайных преобразований. При повторном запуске ячейки аугментации могут отличаться — это ожидаемо.

In [ ]:
import matplotlib.pyplot as plt

images, labels = next(iter(aug_train_loader))

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for ax, image, label in zip(axes.flat, images[:8], labels[:8]):
    image = (image.cpu() * std + mean).clamp(0, 1)
    ax.imshow(image.permute(1, 2, 0))
    ax.set_title(class_names[label.item()])
    ax.axis("off")

plt.tight_layout()
plt.show()
